In [0]:
from pyspark.sql.functions import floor, rand, date_add, lit, when, col as spark_col

num_rows = 10
num_iterations = 1  # Number of sets of columns to create; adjust as needed

df = spark.range(num_rows).toDF("row_id")

# Generate random date offset (0-9 days) for 10 different dates
df = df.withColumn("date_offset",
    floor(rand() * 60).cast("int")  #60
).withColumn("tm_id",
    date_add(lit("2024-10-20"), spark_col("date_offset"))
)

# Generate actv_mem_rltn_ind_cnt with binary distribution (only 0 or 1)
# 70% = 0, 30% = 1 for better clustering separation
print("=== GENERATING BINARY actv_mem_rltn_ind_cnt (0 or 1) ===")
df = df.withColumn("actv_mem_rltn_ind_cnt",
    when(rand() < 0.7, 0).otherwise(1)
)

df.show()

df.createOrReplaceTempView("test_data")

In [0]:
# Fake data generation for testing joins between CUST and MRKT_UNIT tables
# CUST has many-to-one relationship with MRKT_UNIT (multiple customers per household)
from pyspark.sql.functions import rand, when, floor, concat, lpad, col as spark_col, lit, date_add, to_date, expr, current_timestamp  
import string

# Define the character pool and repeat it to make a long string
char_pool = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
long_char_string = char_pool * 20  # 1240 characters

# ============================================================================
# CREATE MRKT_UNIT (HOUSEHOLD) TABLE - The "one" side of the relationship
# ============================================================================
print("=== CREATING MRKT_UNIT (HOUSEHOLD) TABLE ===")

num_mrkt_units = 10  # 1.5 billion households
num_iterations_mrkt = 1  # Number of column sets for MRKT_UNIT

# Create base MRKT_UNIT dataframe
df_mrkt_unit = spark.range(num_mrkt_units).toDF("MRKT_UNIT_BUSN_ID")

# Generate random date offset (0-59 days) for date variety
df_mrkt_unit = df_mrkt_unit.withColumn("date_offset",
    floor(rand() * 60).cast("int")
).withColumn("TM_ID",
    date_add(lit("2024-10-20"), spark_col("date_offset"))
)

# Add household-specific columns
df_mrkt_unit = df_mrkt_unit.withColumn("HOUSEHOLD_SIZE",
    floor(rand() * 6 + 1).cast("int")  # 1-6 members
).withColumn("HOUSEHOLD_INCOME_TIER",
    floor(rand() * 5 + 1).cast("int")  # 1-5 income tiers
).withColumn("HOUSEHOLD_TYPE_IND",
    when(rand() < 0.6, 0).otherwise(1)  # 60%/40% distribution
)


# Drop the temporary date_offset column
df_mrkt_unit = df_mrkt_unit.drop("date_offset")

print("MRKT_UNIT DataFrame created:")
df_mrkt_unit.show(10, truncate=True)
df_mrkt_unit.printSchema()
print(f"MRKT_UNIT row count: {df_mrkt_unit.count()}")

# ============================================================================
# CREATE CUST (CUSTOMER/MEMBER) TABLE - The "many" side of the relationship
# ============================================================================
print("\n=== CREATING CUST (CUSTOMER/MEMBER) TABLE ===")

num_customers = 45  # 4.5 billion customers (multiple per household)
num_iterations_cust = 1  # Number of column sets for CUST

# Create base CUST dataframe
df_cust = spark.range(num_customers).toDF("CUST_BUSN_ID")

# Create many-to-one relationship: multiple customers per household
# Map customers to households (on average ~3 customers per household)
df_cust = df_cust.withColumn("MRKT_UNIT_BUSN_ID",
    floor(rand() * num_mrkt_units).cast("long")
)

# Generate random date offset (0-59 days) - must match MRKT_UNIT for proper joins
df_cust = df_cust.withColumn("date_offset",
    floor(rand() * 60).cast("int")
).withColumn("TM_ID",
    date_add(lit("2024-10-20"), spark_col("date_offset"))
)

# Add customer-specific columns
df_cust = df_cust.withColumn("CUST_AGE",
    floor(rand() * 80 + 18).cast("int")  # Age 18-97
).withColumn("CUST_GENDER",
    when(rand() < 0.5, 0).otherwise(1)  # 50%/50% distribution
).withColumn("ACTV_MEM_RLTN_IND_CNT",
    when(rand() < 0.7, 0).otherwise(1)  # 70%/30% distribution
).withColumn("MEMBER_TYPE",
    floor(rand() * 4 + 1).cast("int")  # 1-4 member types
)


# Drop the temporary date_offset column
df_cust = df_cust.drop("date_offset")

print("CUST DataFrame created:")
df_cust.show(10, truncate=True)
df_cust.printSchema()
print(f"CUST row count: {df_cust.count()}")

# ============================================================================
# CREATE TEMP VIEWS FOR SQL ACCESS
# ============================================================================
print("\n=== CREATING TEMP VIEWS ===")
df_mrkt_unit.createOrReplaceTempView("mrkt_unit_temp")
df_cust.createOrReplaceTempView("cust_temp")

print("✓ Created temp view: mrkt_unit_temp")
print("✓ Created temp view: cust_temp")
print("\nYou can now join these tables using:")
print("  JOIN KEY: MRKT_UNIT_BUSN_ID and TM_ID")
print("  RELATIONSHIP: CUST (many) -> MRKT_UNIT (one)")

In [0]:
%sql
show catalogs;
use catalog kraft_testing;
SHOW SCHEMAS IN kraft_testing;
CREATE OR REPLACE TABLE test_schema.mrkt_unit AS
SELECT * FROM mrkt_unit_temp;

CREATE OR REPLACE TABLE test_schema.cust AS
SELECT * FROM cust_temp;